# SpamShield AI - QLoRA Fine-Tuning with Unsloth

Efficient fine-tuning of Llama 3.2 (or similar) using QLoRA on Enron email dataset.

**Run on Colab with GPU (T4 or better).**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-deps trl peft accelerate bitsandbytes

In [ ]:
import torch
from unsloth import FastLanguageModel
import pandas as pd
import numpy as np
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, classification_report

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Configuration

In [ ]:
CONFIG = {
    'model_name': 'unsloth/Meta-Llama-3.2-1B-Instruct',
    'max_seq_length': 512,
    'lora_rank': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'batch_size': 8,
    'epochs': 3,
    'learning_rate': 2e-4,
    'warmup_ratio': 0.1,
    'seed': 42,
    'save_dir': '/content/drive/My Drive/data/spamshield_lora'
}

print(f'Model: {CONFIG["model_name"]}')
print(f'LoRA rank: {CONFIG["lora_rank"]}, alpha: {CONFIG["lora_alpha"]}')

## 2. Load Data

In [ ]:
DATA_PATH = '/content/drive/My Drive/data/sms_spam.parquet'

df = pd.read_parquet(DATA_PATH)
print(f'Total: {len(df):,}')
print(f'Ham: {(df["label"] == 0).sum():,} ({(df["label"] == 0).mean()*100:.1f}%)')
print(f'Spam: {(df["label"] == 1).sum():,} ({(df["label"] == 1).mean()*100:.1f}%)')
df.head()

In [ ]:
# Use all SMS data (only ~5.5k rows)
df_sample = df.reset_index(drop=True)
print(f'Using all {len(df_sample):,} samples')

## 3. Format for Instruction Tuning

In [ ]:
SPAM_TEMPLATE = "Classify the following SMS message as spam or ham.\n\nMessage:\n{email}\n\nClassification:"

def format_example(row):
    label = "spam" if row['label'] == 1 else "ham"
    text = str(row['text'])[:500]
    return {
        'text': SPAM_TEMPLATE.format(email=text),
        'output': label
    }

formatted = df_sample.apply(format_example, axis=1, result_type='expand')
print('Sample formatted example:')
print(formatted.iloc[0]['text'][:300])
print(f'Output: {formatted.iloc[0]["output"]}')

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(formatted, test_size=0.15, random_state=CONFIG['seed'])

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

print(f'Train: {len(train_dataset):,}')
print(f'Test: {len(test_dataset):,}')

## 4. Load Model with Unsloth

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    dtype=None,
    load_in_4bit=True,
)

print(f'Model loaded: {CONFIG["model_name"]}')

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG['lora_rank'],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG['seed'],
)

model.print_trainable_parameters()

## 5. Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=CONFIG['max_seq_length'],
    args=TrainingArguments(
        per_device_train_batch_size=CONFIG['batch_size'],
        gradient_accumulation_steps=4,
        num_train_epochs=CONFIG['epochs'],
        learning_rate=CONFIG['learning_rate'],
        warmup_ratio=CONFIG['warmup_ratio'],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=50,
        output_dir="outputs",
        optim="adamw_8bit",
        seed=CONFIG['seed'],
    ),
)

print('Starting training...')

In [ ]:
trainer_stats = trainer.train()
print(f'\nTraining complete!')
print(f'Training loss: {trainer_stats.training_loss:.4f}')
print(f'Training time: {trainer_stats.metrics["train_runtime"]:.0f}s')

## 6. Test Predictions

In [ ]:
FastLanguageModel.for_inference(model)

def predict(text):
    prompt = SPAM_TEMPLATE.format(email=str(text)[:1500])
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=CONFIG['max_seq_length']).to(DEVICE)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.1)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split('Classification:')[-1].strip().lower()
    if 'spam' in answer:
        return 1, 0.95
    else:
        return 0, 0.95

## 7. Evaluate on Test Set

In [ ]:
# Evaluate on a sample
EVAL_SIZE = min(500, len(test_df))
eval_df = test_df.sample(EVAL_SIZE, random_state=CONFIG['seed'])

y_true, y_pred = [], []
for _, row in eval_df.iterrows():
    true_label = 1 if row['output'] == 'spam' else 0
    pred_label, _ = predict(row['text'].split('Email:\n')[1].split('\n\nClassification:')[0])
    y_true.append(true_label)
    y_pred.append(pred_label)

print(classification_report(y_true, y_pred, target_names=['Ham', 'Spam']))

## 8. Save LoRA Adapter to Drive

In [ ]:
import os
import json

save_dir = CONFIG['save_dir']
!mkdir -p $save_dir

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

results = {
    'accuracy': accuracy_score(y_true, y_pred),
    'f1': f1_score(y_true, y_pred),
    'config': CONFIG,
    'training_loss': trainer_stats.training_loss,
}
with open(f'{save_dir}/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'LoRA adapter saved to {save_dir}')
adapter_size = sum(os.path.getsize(os.path.join(save_dir, f)) for f in os.listdir(save_dir)) / 1e6
print(f'Adapter size: {adapter_size:.1f} MB')

## 9. Export to GGUF (Optional - for Ollama/llama.cpp)

In [ ]:
# Uncomment to export as GGUF (useful for local inference)
# model.save_pretrained_gguf(f'{save_dir}/gguf', tokenizer, quantization_method="q4_k_m")
# print('GGUF exported!')

## 10. Merge & Export for SageMaker

In [ ]:
import tarfile

# Merge LoRA into base model
merged_model = model.merge_and_unload()

# Save merged model
merged_dir = f'{save_dir}/merged'
!mkdir -p $merged_dir
merged_model.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)

# Create tar.gz for SageMaker
tar_path = f'{save_dir}/model.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    for root, dirs, files in os.walk(merged_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, merged_dir)
            tar.add(file_path, arcname=arcname)

tar_size = os.path.getsize(tar_path) / 1e6
print(f'Merged model saved to {merged_dir}')
print(f'tar.gz: {tar_path} ({tar_size:.1f} MB)')

In [ ]:
# Upload to S3 (uncomment and set your bucket)
# import boto3
#
# s3 = boto3.client('s3')
# BUCKET = 'spamshield-ai-models'
# S3_KEY = 'model/model.tar.gz'
#
# s3.upload_file(tar_path, BUCKET, S3_KEY)
# print(f'Uploaded to s3://{BUCKET}/{S3_KEY}')